# Lab 01. Exploring Data Representations

# Overview

A dataset does not have only one useful representation.

In this lab, we examine how the same underlying data can be represented as
records, sequences, matrices, vectors, and graphs.

> **Before choosing an algorithm, choose a representation.**

# Part 1. Record Data

We use the **Titanic dataset** to examine how the same passenger records
change when we represent them differently.

The Titanic dataset contains passenger information such as class, sex, age,
fare, and survival status. For this lab, we select a subset of numerical and
categorical attributes.

In [ ]:
#| label: setup-record
#| include: false

from pathlib import Path
import sys

_lab = Path("exercises/lab01")
if not (_lab / "lab01_setup.py").exists():
    _lab = Path(".")
sys.path.insert(0, str(_lab.resolve()))

import lab01_setup

import numpy as np
import pandas as pd

from sklearn.preprocessing import MultiLabelBinarizer

from lab01_record import (
    fill_titanic_missing_values,
    build_transactions,
    build_bipartite_graph,
    plot_bipartite_graph,
)

pd.set_option("display.max_colwidth", 100)

## 1.1 Load the Data

`load_titanic()` reads `data/record/titanic.csv`. If that file is missing, it
downloads the seaborn Titanic table and saves it there.

In [ ]:
from data.loader import load_titanic

titanic_raw = load_titanic()

type(titanic_raw), titanic_raw.shape

In [ ]:
titanic_raw.columns.tolist()

In [ ]:
selected_columns = [
    "survived",
    "pclass",
    "sex",
    "age",
    "fare",
    "embarked",
    "alone",
]

titanic = titanic_raw[selected_columns].copy()

type(titanic), titanic.shape

In [ ]:
titanic.head()

In [ ]:
titanic["survived"].value_counts()

## 1.2 Representation 1: Numerical Feature Matrix

The same passenger records can be represented as **numerical vectors**.

`survived` is excluded because it represents the outcome rather than an input
feature.

In [ ]:
X_numeric = titanic.drop(
    columns=["survived"]
).copy()

Missing values in `age` and `embarked` are filled before conversion.

In [ ]:
X_numeric = fill_titanic_missing_values(
    X_numeric
)

`pd.get_dummies()` converts categorical values into binary (0/1) indicator
columns.

In [ ]:
X_numeric = pd.get_dummies(
    X_numeric,
    columns=[
        "pclass",
        "sex",
        "embarked",
        "alone",
    ],
    dtype=int,
)

type(X_numeric), X_numeric.shape

In [ ]:
X_numeric.head()

> #### 💡 Tip
> Is this a different dataset?  
> No. The passengers are the same. Only their representation has changed from
> mixed data types to numerical values.

## 1.3 Representation 2: Transaction Representation

The same passenger can also be represented as a **set of attribute-value
items**.

`pd.cut()` and `pd.qcut()` discretize continuous values into categories.

In [ ]:
titanic_tx = fill_titanic_missing_values(
    titanic.copy()
)

titanic_tx["age_group"] = pd.cut(
    titanic_tx["age"],
    bins=[0, 12, 18, 35, 60, np.inf],
    labels=[
        "child",
        "teen",
        "young_adult",
        "adult",
        "senior",
    ],
)

titanic_tx["fare_group"] = pd.qcut(
    titanic_tx["fare"],
    q=4,
    labels=[
        "low",
        "medium",
        "high",
        "very_high",
    ],
)

`build_transactions()` converts each passenger into a set of attribute-value
items.

In [ ]:
transactions = build_transactions(
    titanic_tx
)

type(transactions), type(transactions.iloc[0])

In [ ]:
for i, items in transactions.head().items():
    print(i, ":", items)

> #### 💡 Tip
> What changed from the original record representation?  
> The passenger is now represented as a set of attribute-value items.

## 1.4 Representation 3: Binary Item Matrix

The transaction representation can be converted into a **binary matrix**.

`MultiLabelBinarizer` converts each passenger's set of items into binary
(0/1) columns, one for each possible item.

In [ ]:
mlb = MultiLabelBinarizer()

X_items = mlb.fit_transform(
    transactions
)

item_matrix = pd.DataFrame(
    X_items,
    index=transactions.index,
    columns=mlb.classes_,
)

type(item_matrix), item_matrix.shape

In [ ]:
item_matrix.head()

The matrix has the following meaning:

- Row = passenger
- Column = attribute-value item
- Value = 1 if the passenger has that item
- Value = 0 if the passenger does not have that item

> #### 💡 Tip
> Does the binary matrix contain new information?  
> No. It represents the same transaction information in matrix form.

## 1.5 Representation 4: Passenger-Attribute Bipartite Graph

The same transaction data can also be represented as a **bipartite graph**.

We create two kinds of nodes:

- Passenger nodes
- Attribute-value nodes

An edge connects a passenger to an attribute-value item that belongs to that
passenger.

In [ ]:
G = build_bipartite_graph(
    transactions,
    n_show=6,
)

type(G)

In [ ]:
G.number_of_nodes(), G.number_of_edges()

In [ ]:
list(G.edges())[:10]

In [ ]:
#| fig-cap: Passenger-attribute bipartite graph

plot_bipartite_graph(G)